In [31]:
import pandas as pd
from helpers import get_factor, get_price

In [32]:
pd.set_option('mode.chained_assignment',  None) 

In [33]:
CDF = pd.read_csv("../production/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

In [34]:
smard = pd.read_csv("Gro_handelspreise_202401010000_202501010000_Stunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

In [35]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [36]:
CDF2 = CDF.loc[CDF.produced_at > "2023-12-31 23:50"].loc[CDF.produced_at < "2025-01-01 00:00"]

In [37]:
dataset = CDF2.merge(seem, left_on="variable", right_on="sseid")

In [38]:
magic = dataset.groupby(["produced_at", "plantid"]).sum()

In [39]:
magic2 = magic[["value"]]

In [40]:
#magic2.sort_values(["produced_at", "value"])

In [41]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [42]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")

In [61]:
merged

,produced_at,plantid,value,timestamp,price,revenue
0,2024-01-01 00:00:00,06-02-B10117A007,68,2024-01-01 00:00:00,0.10,6.8
1,2024-01-01 00:00:00,BB23020490,156,2024-01-01 00:00:00,0.10,15.6
2,2024-01-01 00:00:00,BB45025564,894,2024-01-01 00:00:00,0.10,89.4
3,2024-01-01 00:00:00,BB45025611,267,2024-01-01 00:00:00,0.10,26.7
4,2024-01-01 00:00:00,BE166928,167,2024-01-01 00:00:00,0.10,16.7
...,...,...,...,...,...,...
474331,2024-12-31 23:00:00,SL0101003-G,0,2024-12-31 23:00:00,0.52,0.0
474332,2024-12-31 23:00:00,SL0105352-G,0,2024-12-31 23:00:00,0.52,0.0
474333,2024-12-31 23:00:00,SN70015796,0,2024-12-31 23:00:00,0.52,0.0
474334,2024-12-31 23:00:00,SN80011277,0,2024-12-31 23:00:00,0.52,0.0


In [43]:
merged["revenue"] = merged["value"] * merged["price"]
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [44]:
tmp1.reset_index(inplace=True)

In [45]:
revenue = tmp1[["plantid", "revenue"]]

In [46]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [47]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [48]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [49]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

,year,plantid,pollutant,releases_to,amount,potency,unit_2,amount_2,pollutant2


In [50]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [51]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [52]:
tmp2 = tmp1

In [53]:
#tmp2

In [54]:
coal_cost_per_t = 103.5 or 120
co2_cost = 80
#electricity_price = 78.50

In [55]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [56]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [57]:
co2s3.dtypes

plantid      object
amount_2    float64
dtype: object

In [58]:
tmp2.sort_values("profit")

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit


In [59]:
#tmp2["profit_adj"] = (tmp2["revenue"] * 1.10) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [60]:
tmp2.sort_values("profit")

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit
